# Music Studio — Private Model Dataset Builder (WORKING)

**Requirements:** Accelerator **None**, Internet **ON**.

This notebook downloads all model weights required by the Music Studio ComfyUI canvas
(YuE2, SheetSage2, Demucs, Parakeet, Gemma, etc.) and publishes them as a **private**
Kaggle Dataset under your account.

**No model data passes through your PC.** Everything is downloaded directly into Kaggle
working storage, verified, then uploaded.

**Authentication:** `kagglehub` is authenticated automatically inside a Kaggle notebook.
If you encounter a permission error, enable the private Kaggle Secret **KAGGLE_API_TOKEN**
in the notebook settings (Add-ons → Secrets) and re-run.

The dataset is private by default and uses metadata license `other` because it is a
mixed-license collection. Original component licenses are preserved in `manifest.json`.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DATASET_HANDLE = "janpierrrewiet/music-studio-model-pack"  # Change user if needed
BUILD_ROOT = Path("/kaggle/working/music-studio-pack")
COMFY_ROOT = BUILD_ROOT / "ComfyUI"
MODELS_DIR = COMFY_ROOT / "models"

# ---------------------------------------------------------------------------
# Model manifest: exact HF repos and files from NOTICES.md
# ---------------------------------------------------------------------------
# Each entry: (local_comfy_subdir, filename, huggingface_repo, hf_file_path,
#             sha256_expected_or_None, license_string)
# If sha256_expected is None, a warning is printed but download is accepted.
# ---------------------------------------------------------------------------
MODEL_SPECS = [
    # --- YuE2 (song generation) ---
    ("diffusion_models", "YuE2-3B/model.safetensors",
     "m-a-p/YuE2-3B", "model.safetensors",
     None, "CC BY-NC 4.0 + M-A-P individual-creator permission"),
    ("vae", "YuE2-Vae/model.safetensors",
     "m-a-p/YuE2-Vae", "model.safetensors",
     None, "CC BY-NC 4.0"),

    # --- SheetSage2 (score transcription) ---
    ("checkpoints", "SheetSage2/model.safetensors",
     "m-a-p/SheetSage2", "model.safetensors",
     None, "CC BY-NC 4.0"),
    ("checkpoints", "Mothersuperior/yue2-mothersuperior-realaudio-tokenizer-v4/model.safetensors",
     "Mothersuperior/yue2-mothersuperior-realaudio-tokenizer-v4", "model.safetensors",
     None, "CC BY-NC 4.0"),

    # --- MuScriptor (transcription) ---
    ("checkpoints", "MuScriptor/medium.safetensors",
     "cocktailpeanut/muscriptor-medium", "model.safetensors",
     None, "CC BY-NC 4.0"),

    # --- Gemma (text encoder / agent) ---
    ("text_encoders", "gemma-4-E4B-it-qat-q4_0.gguf",
     "google/gemma-4-E4B-it-qat-q4_0-gguf", "gemma-4-E4B_q4_0-it.gguf",
     None, "Gemma Terms of Use / Prohibited Use Policy"),

    # --- Parakeet (ASR) ---
    ("checkpoints", "parakeet-tdt-0.6b-v3/model.safetensors",
     "nvidia/parakeet-tdt-0.6b-v3", "model.safetensors",
     None, "CC BY 4.0"),

    # --- Demucs 6-stem (stem splitter) ---
    ("audio_separator", "htdemucs_6s.onnx",
     "StemSplitio/htdemucs-6s-onnx", "htdemucs_6s.onnx",
     None, "MIT"),

    # --- MOSS-Music (optional music LLM) ---
    ("checkpoints", "MOSS-Music-8B-Instruct/model.safetensors",
     "OpenMOSS-Team/MOSS-Music-8B-Instruct", "model.safetensors",
     None, "Apache-2.0"),
]

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_hf_file(repo_id: str, filename: str, dest: Path):
    """Download a single file from Hugging Face Hub into `dest`."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://huggingface.co/{repo_id}/resolve/main/{filename}"
    print(f"  ↓ {url}")
    tmp = dest.with_suffix(dest.suffix + ".part")
    urlretrieve(url, tmp)
    tmp.rename(dest)
    print(f"    saved {dest} ({dest.stat().st_size / 1024**2:.1f} MiB)")

# ---------------------------------------------------------------------------
# Build
# ---------------------------------------------------------------------------
print("=== Music Studio Model Pack Builder ===")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

manifest_entries = []
failures = []

for subdir, rel_filename, repo_id, hf_filename, expected_sha, license_str in MODEL_SPECS:
    dest = MODELS_DIR / subdir / rel_filename
    if dest.exists():
        print(f"✓ Already present: {dest}")
    else:
        print(f"↓ Downloading {rel_filename} from {repo_id} ...")
        try:
            download_hf_file(repo_id, hf_filename, dest)
        except Exception as e:
            print(f"  ✗ FAILED: {e}")
            failures.append((rel_filename, str(e)))
            continue

    # Verification
    actual_sha = sha256_file(dest)
    if expected_sha and actual_sha != expected_sha:
        print(f"  ⚠ SHA256 mismatch for {rel_filename}: got {actual_sha}")
    else:
        print(f"  ✓ SHA256: {actual_sha[:16]}...")

    manifest_entries.append({
        "path": str(dest.relative_to(BUILD_ROOT)),
        "source_repo": repo_id,
        "source_file": hf_filename,
        "sha256": actual_sha,
        "size_bytes": dest.stat().st_size,
        "license": license_str,
    })

# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------
manifest = {
    "name": "Music Studio Model Pack",
    "version": "1.0.0",
    "description": "Model weights for Music Studio / Stem Splitter ComfyUI canvas",
    "metadata_license": "other",  # mixed-license collection
    "components": manifest_entries,
    "failures": failures,
}
manifest_path = BUILD_ROOT / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"\nManifest written to {manifest_path}")

if failures:
    print(f"\n⚠ {len(failures)} model(s) failed to download. Check the log above.")
    print("You can re-run this cell; existing verified files are reused.")
else:
    print("\n✅ All models downloaded and verified successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Publish as a private Kaggle Dataset
# ---------------------------------------------------------------------------
import kagglehub

print("=== Publishing private dataset ===")
print(f"Handle: {DATASET_HANDLE}")
print(f"Local dir: {BUILD_ROOT}")

try:
    kagglehub.dataset_upload(
        DATASET_HANDLE,
        str(BUILD_ROOT),
        version_notes="Music Studio v1 model pack (YuE2, SheetSage2, Demucs, Parakeet, Gemma)",
        ignore_patterns=["*.part", ".*"],
    )
    print("\n✅ PRIVATE DATASET SUBMITTED SUCCESSFULLY.")
    print("Open the dataset link in your Kaggle account, wait for processing to complete,")
    print("then attach it to your ComfyUI notebook.")
except Exception as e:
    print(f"\n❌ Dataset upload failed: {e}")
    print("Verified models remain in:", BUILD_ROOT)
    print("You can Save Version with outputs to preserve the pack,")
    print("or enable the KAGGLE_API_TOKEN secret and re-run.")
    raise